In [1]:
# Libraries
import caveclient
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def create_copy_paste_annot_list(annot_pt_list = [], mip=(32,32,45), description_list = None, connected=False):
    """ list of annotations to copy/paste to spleunker.cave """
    if description_list == None:
        description_list = ["" for _ in range(len(annot_pt_list))]
    else:
        assert len(annot_pt_list) == len(description_list)

    annots = []
    for i, pt in enumerate(annot_pt_list):
        if connected:
            pt_dict = {
              "pointA": list([int(pt[0][0]/mip[0]), int(pt[0][1]/mip[1]), int(pt[0][2]/mip[2])]),
              "pointB": list([int(pt[1][0]/mip[0]), int(pt[1][1]/mip[1]), int(pt[1][2]/mip[2])]),
              "type": "line",
              "id": "a6f2611ba4basd3635da2ea158ee9853fab0da"+str(i), # random id. Needs to be unique for each annot
            }
        else:
            pt_dict = {"point": list([int(pt[0]/mip[0]), int(pt[1]/mip[1]), int(pt[2]/mip[2])]), "description": str(description_list[i]), "type": "point", "id": "b6f2611ba4basd3635da2ea158ee9853fab0dabf"+str(i)}
            
        annots.append(pt_dict)
    subprocess.run("pbcopy", text=True, input=json.dumps(annots))
    return annots

In [3]:
# Setup caveclient
client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr", ) #server_address='https://global.daf-apis.com')

In [83]:
# Synapses df
cltmr_synapses_df = client.materialize.query_table('sensory_neuron_synapses', filter_equal_dict = {'post_pt_root_id': 720575940883734520})

In [84]:
cltmr_synapses_df

,id,created,superceded_id,valid,size,pre_pt_supervoxel_id,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id,pre_pt_position,post_pt_position,ctr_pt_position
0,84484,2025-07-04 19:57:37.780660+00:00,NaN,t,127,76138982408280085,720575940899184211,76138982408284983,720575940883734520,"[38337, 4408, 2456]","[38343, 4420, 2463]","[38346, 4412, 2459]"
1,15936,2025-07-03 21:02:05.705166+00:00,NaN,t,20,74661375749804737,720575940915691330,74591007005603656,720575940883734520,"[27139, 5516, 1627]","[27124, 5497, 1627]","[27133, 5503, 1627]"
2,15937,2025-07-03 21:02:05.706084+00:00,NaN,t,103,74591007072627761,720575940897699138,74591007072623864,720575940883734520,"[27060, 5533, 1648]","[27055, 5516, 1644]","[27062, 5528, 1643]"
3,15938,2025-07-03 21:02:05.706827+00:00,NaN,t,26,74661375816819370,720575940886010485,74661375816821902,720575940883734520,"[27186, 5504, 1673]","[27165, 5505, 1670]","[27179, 5518, 1672]"
4,15939,2025-07-03 21:02:05.707585+00:00,NaN,t,156,74661375816812697,720575940786826250,74661375816815157,720575940883734520,"[27144, 5521, 1659]","[27138, 5502, 1661]","[27149, 5509, 1661]"
5,15940,2025-07-03 21:02:05.708346+00:00,NaN,t,36,74661375749757950,720575940915691330,74661375749763088,720575940883734520,"[27452, 5556, 1587]","[27455, 5573, 1592]","[27457, 5568, 1589]"
6,15941,2025-07-03 21:02:05.709113+00:00,NaN,t,31,75294557142608786,720575940890452150,75294625862139034,720575940883734520,"[31756, 4597, 1851]","[31758, 4621, 1847]","[31760, 4611, 1849]"
7,15942,2025-07-03 21:02:05.709849+00:00,NaN,t,137,75294557209675347,720575940823547264,75294557209674013,720575940883734520,"[32030, 4485, 1924]","[32018, 4466, 1920]","[32028, 4475, 1921]"
8,15943,2025-07-03 21:02:05.710612+00:00,NaN,t,45,75294557209675351,720575940823547264,75294557209668438,720575940883734520,"[32054, 4488, 1921]","[32052, 4465, 1916]","[32059, 4480, 1917]"
9,15944,2025-07-03 21:02:05.711388+00:00,NaN,t,147,75927876511238148,720575940856573381,75927876511219355,720575940883734520,"[36512, 4500, 3054]","[36496, 4509, 3051]","[36512, 4502, 3048]"


In [85]:
# Create a coordinates list
coordinates = []
for i, row in cltmr_synapses_df.iterrows():
    coordinates.append([row['pre_pt_position'], row['post_pt_position']])  

In [86]:
# Use util function to create copy/past version of annots ... paste to NG json file # mip=(2,2,1)
annots = create_copy_paste_annot_list(annot_pt_list = coordinates, mip=(2,2,1), connected=True)

In [4]:
import os
import json
import pandas as pd

# Assuming `client` is already authenticated and imported
# from caveclient import CAVEclient
# client = CAVEclient("your_cave_project")

def find_synapse_annotations(data):
    """Recursively find all synapse-like entries in a JSON structure."""
    matches = []

    def recurse(node):
        if isinstance(node, list):
            for item in node:
                recurse(item)
        elif isinstance(node, dict):
            if (
                "pointA" in node and
                "pointB" in node and
                node.get("type") == "line"
            ):
                matches.append(node)
            for value in node.values():
                recurse(value)

    recurse(data)
    return matches

# Folder with the annotation JSON files
folder = "/Users/wangchu/Downloads/synapse_prediction_accuracy/batch_2"
all_ids_to_delete = []

for filename in os.listdir(folder):
    if not filename.endswith(".json"):
        continue

    file_path = os.path.join(folder, filename)
    root_id = int(os.path.splitext(filename)[0])  # root_id from filename

    with open(file_path, 'r') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Error reading {filename}: {e}")
            continue

    # Get synapse annotations and find indices to delete
    synapses = find_synapse_annotations(data)
    indices_to_delete = [i for i, item in enumerate(synapses)
                         if item.get("description") == "f"]

    if not indices_to_delete:
        continue

    # Query synapses table for this root ID
    try:
        syn_df = client.materialize.query_table(
            'sensory_neuron_synapses',
            filter_equal_dict={'post_pt_root_id': root_id}
        )
    except Exception as e:
        print(f"Query error for root ID {root_id}: {e}")
        continue

    # Make sure index list is valid
    if max(indices_to_delete) >= len(syn_df):
        print(f"Index out of range in file {filename}, skipping")
        continue

    # Extract 'id' values for synapses to delete
    ids_to_delete = syn_df.iloc[indices_to_delete]['id'].tolist()
    all_ids_to_delete.extend(ids_to_delete)

# Final list of all synapse IDs to delete
print(f"\nTotal synapse IDs to delete: {len(all_ids_to_delete)}")



Total synapse IDs to delete: 641


In [5]:
all_ids_to_delete

[84484,
 15936,
 15937,
 15941,
 15942,
 15944,
 15945,
 15946,
 15947,
 15948,
 15949,
 15950,
 15952,
 15953,
 15954,
 15955,
 15956,
 15957,
 15958,
 15959,
 15960,
 15961,
 15962,
 15963,
 15964,
 15965,
 15966,
 15967,
 15968,
 15969,
 15970,
 15971,
 15972,
 15973,
 15974,
 15975,
 15976,
 15603,
 15604,
 15605,
 15606,
 15607,
 15608,
 15609,
 15610,
 15611,
 15612,
 15613,
 15614,
 15615,
 15616,
 15617,
 15618,
 15619,
 15620,
 15621,
 15622,
 15623,
 15624,
 15625,
 15626,
 15627,
 15629,
 15630,
 15631,
 15632,
 15633,
 15634,
 15636,
 15637,
 15638,
 15639,
 15640,
 15641,
 15642,
 15643,
 15644,
 15645,
 15646,
 15647,
 15648,
 15649,
 15650,
 15651,
 15652,
 15653,
 15654,
 15655,
 13170,
 13000,
 13001,
 13003,
 13007,
 13012,
 13013,
 13019,
 13020,
 13021,
 13032,
 13042,
 13077,
 13082,
 13086,
 13088,
 13094,
 13095,
 13103,
 13113,
 13127,
 13146,
 13157,
 13163,
 13164,
 13168,
 13175,
 13198,
 13200,
 13201,
 13211,
 12818,
 13242,
 13244,
 13268,
 13269,
 13274,


In [6]:
client.annotation.delete_annotation(table_name='sensory_neuron_synapses',annotation_ids=all_ids_to_delete)

[418,
 422,
 429,
 431,
 443,
 449,
 451,
 452,
 458,
 460,
 467,
 469,
 470,
 474,
 477,
 478,
 480,
 483,
 484,
 485,
 504,
 510,
 515,
 521,
 522,
 523,
 527,
 532,
 543,
 548,
 552,
 554,
 560,
 576,
 577,
 590,
 602,
 632,
 637,
 640,
 644,
 645,
 651,
 659,
 704,
 726,
 732,
 755,
 761,
 777,
 779,
 796,
 800,
 802,
 808,
 813,
 822,
 833,
 835,
 839,
 851,
 857,
 861,
 864,
 880,
 883,
 884,
 894,
 895,
 902,
 906,
 913,
 917,
 920,
 921,
 922,
 928,
 933,
 936,
 938,
 941,
 945,
 3064,
 3066,
 3069,
 3074,
 3077,
 3085,
 3090,
 3094,
 3096,
 3100,
 3101,
 3102,
 3103,
 3107,
 3108,
 3110,
 3111,
 3118,
 3121,
 3124,
 3125,
 3136,
 3140,
 3143,
 3144,
 3146,
 3147,
 3158,
 3159,
 3167,
 3169,
 3170,
 3171,
 3177,
 3180,
 3183,
 3189,
 3190,
 3195,
 3196,
 3200,
 3204,
 3206,
 3208,
 3217,
 3218,
 3222,
 3229,
 3231,
 3234,
 3235,
 3237,
 3238,
 3246,
 3249,
 3251,
 3255,
 3256,
 3259,
 3260,
 3262,
 3263,
 3265,
 3267,
 3275,
 3280,
 3282,
 3283,
 3284,
 3286,
 3289,
 3291,
 329